# Practical audit of candidate recharge wells: Faisalabad case study

This notebook takes 100 candidate recharge-well sites and:

1. checks them against OpenStreetMap roads, canals and drains,
2. records a satellite check done by hand,
3. moves problem wells to nearby clear locations (two rounds),
4. writes a final well list and a map.

**Scope:** practical site screening only (canals, drains, roads, buildings). It is not a
hydrogeological or groundwater-quality suitability assessment.

**Needs in Google Drive, folder `FSD_Raw_Export`:**
`Static_Predictors.tif`, `Recommended_Well_Locations_FINAL.csv` (columns `lon`, `lat`),
`Faisalabad_Drainage_Network.shp`, `Faisalabad_Settlements_Vector.shp`,
`Faisalabad_Boundary.shp` (for the map). Run the cells from top to bottom.

In [ ]:
!pip install -q geopandas shapely pyproj rasterio requests

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import requests
import matplotlib.pyplot as plt
from shapely.geometry import LineString, Point, Polygon
from shapely.ops import unary_union

from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/FSD_Raw_Export/'

## 1. Functions

In [ ]:
def normalize(arr, lo, hi, invert=False):
    """Scale to 0..1 with fixed limits. invert=True flips the scale."""
    scaled = np.clip((np.asarray(arr, dtype=float) - lo) / (hi - lo), 0.0, 1.0)
    return 1.0 - scaled if invert else scaled


def weighted_index(layers, weights):
    """Weighted sum of layers. Weights are rescaled to add up to 1."""
    w = np.array(list(weights.values()), dtype=float)
    w = w / w.sum()
    out = np.zeros(np.shape(layers[next(iter(weights))]), dtype=float)
    for name, wi in zip(weights, w):
        out = out + wi * np.asarray(layers[name], dtype=float)
    return out


DEFAULT_BUFFER_M = 200.0


def audit_sites(sites, constraints, buffers_m=None, near_miss_m=50.0,
                lon="lon", lat="lat", metric_crs=None):
    """Label each site clear / uncertain / conflict.

    conflict : closer to a constraint than its buffer (default 200 m)
    uncertain: a layer is missing or empty, or the site is within near_miss_m
               beyond a buffer (map data can be off by that much)
    clear    : no conflict found in the layers you gave
    """
    buffers = buffers_m or {}
    result = sites.reset_index(drop=True).copy()
    n = len(result)
    pts = gpd.GeoDataFrame(result[[lon, lat]].copy(),
                           geometry=gpd.points_from_xy(result[lon], result[lat]),
                           crs="EPSG:4326")
    crs = metric_crs or pts.estimate_utm_crs()
    pts_m = pts.to_crs(crs)
    conflict = np.zeros(n, dtype=bool)
    uncertain = np.zeros(n, dtype=bool)
    reasons = [[] for _ in range(n)]
    for name, layer in constraints.items():
        buf = float(buffers.get(name, DEFAULT_BUFFER_M))
        if layer is None or len(layer) == 0:
            result[f"dist_{name}_m"] = np.nan
            uncertain[:] = True
            for r in reasons:
                r.append(f"{name}: no data")
            continue
        layer_m = layer.to_crs(crs)
        joined = gpd.sjoin_nearest(pts_m[["geometry"]], layer_m[["geometry"]],
                                   distance_col="d")
        dist = joined.groupby(level=0)["d"].min().reindex(range(n)).to_numpy()
        result[f"dist_{name}_m"] = dist
        for i, d in enumerate(dist):
            if d < buf:
                conflict[i] = True
                reasons[i].append(f"{name}: {d:.0f} m (limit {buf:.0f} m)")
            elif d < buf + near_miss_m:
                uncertain[i] = True
                reasons[i].append(f"{name}: near limit, {d:.0f} m")
    result["label"] = np.where(conflict, "conflict",
                               np.where(uncertain, "uncertain", "clear"))
    result["reason"] = ["; ".join(r) for r in reasons]
    return result


def summarize_audit(audited):
    counts = audited["label"].value_counts()
    return {k: int(counts.get(k, 0)) for k in ("clear", "uncertain", "conflict")}


def _nearest_m(points_m, layer_m):
    j = gpd.sjoin_nearest(points_m[["geometry"]], layer_m[["geometry"]], distance_col="d")
    return j.groupby(level=0)["d"].min().reindex(range(len(points_m))).to_numpy()


def relocate_sites(sites_to_move, index, transform, constraints, buffers_m=None,
                   other_sites=None, search_radius_m=2000.0, keep_apart_m=2000.0,
                   margin_m=50.0, prefer="nearest", score_drop=0.03,
                   lon="lon", lat="lat"):
    """For each site, look around it for a cell that is at least buffer + margin_m away
    from every constraint, and keep_apart_m away from other wells.

    prefer="nearest": the closest such cell whose score is at least old score - score_drop
                      (if none, the best-scoring such cell, status "moved_lower_score").
    prefer="score"  : the best-scoring such cell.
    index     : 2-D suitability array on the same grid as `transform` (EPSG:4326, north-up)
    transform : affine transform of that grid
    """
    buffers = buffers_m or {}
    s = sites_to_move.reset_index(drop=True)
    idx = np.asarray(index, dtype=float)
    crs = gpd.GeoDataFrame(
        geometry=gpd.points_from_xy(s[lon], s[lat]), crs="EPSG:4326").estimate_utm_crs()
    layers_m = {n: l.to_crs(crs) for n, l in constraints.items()
                if l is not None and len(l) > 0}
    placed = []
    if other_sites is not None and len(other_sites):
        o = gpd.GeoDataFrame(geometry=gpd.points_from_xy(other_sites[lon], other_sites[lat]),
                             crs="EPSG:4326").to_crs(crs)
        placed = [(g.x, g.y) for g in o.geometry]
    rows_out = []
    for i, row in s.iterrows():
        col0 = int((row[lon] - transform.c) / transform.a)
        row0 = int((row[lat] - transform.f) / transform.e)
        mx = 111_320.0 * np.cos(np.deg2rad(row[lat]))
        rx = int(search_radius_m / (abs(transform.a) * mx)) + 1
        ry = int(search_radius_m / (abs(transform.e) * 111_320.0)) + 1
        r0, r1 = max(row0 - ry, 0), min(row0 + ry + 1, idx.shape[0])
        c0, c1 = max(col0 - rx, 0), min(col0 + rx + 1, idx.shape[1])
        rr, cc = np.mgrid[r0:r1, c0:c1]
        rr, cc = rr.ravel(), cc.ravel()
        sc = idx[rr, cc]
        keep = np.isfinite(sc)
        rr, cc, sc = rr[keep], cc[keep], sc[keep]
        lons = transform.c + (cc + 0.5) * transform.a
        lats = transform.f + (rr + 0.5) * transform.e
        cand = gpd.GeoDataFrame({"score": sc}, geometry=gpd.points_from_xy(lons, lats),
                                crs="EPSG:4326").to_crs(crs)
        origin = gpd.GeoSeries(gpd.points_from_xy([row[lon]], [row[lat]]),
                               crs="EPSG:4326").to_crs(crs).iloc[0]
        cx = cand.geometry.x.to_numpy()
        cy = cand.geometry.y.to_numpy()
        ok = np.hypot(cx - origin.x, cy - origin.y) <= search_radius_m
        for name, layer_m in layers_m.items():
            need = float(buffers.get(name, 200.0)) + margin_m
            dist = np.full(len(cand), np.inf)
            dist[ok] = _nearest_m(cand[ok].reset_index(drop=True), layer_m)
            ok &= dist >= need
        for (px, py) in placed:
            ok &= np.hypot(cx - px, cy - py) >= keep_apart_m
        rec = {"old_lon": row[lon], "old_lat": row[lat],
               "old_score": float(idx[min(max(row0, 0), idx.shape[0] - 1),
                                      min(max(col0, 0), idx.shape[1] - 1)])}
        if ok.any():
            dist_o = np.hypot(cx - origin.x, cy - origin.y)
            status = "moved"
            if prefer == "nearest":
                good = ok & (cand["score"].to_numpy() >= rec["old_score"] - score_drop)
                if good.any():
                    best = np.argmin(np.where(good, dist_o, np.inf))
                else:
                    best = np.argmax(np.where(ok, cand["score"].to_numpy(), -np.inf))
                    status = "moved_lower_score"
            else:
                best = np.argmax(np.where(ok, cand["score"].to_numpy(), -np.inf))
            rec.update(new_lon=float(lons[best]), new_lat=float(lats[best]),
                       new_score=float(sc[best]), status=status,
                       moved_m=float(dist_o[best]))
            placed.append((cx[best], cy[best]))
        else:
            rec.update(new_lon=np.nan, new_lat=np.nan, new_score=np.nan,
                       status="no_alternative", moved_m=np.nan)
        rows_out.append(rec)
    return pd.DataFrame(rows_out)


OVERPASS_URL = "https://overpass-api.de/api/interpreter"
OSM_FILTERS = [
    '["highway"~"^(motorway|trunk|primary|secondary|tertiary|unclassified|residential)(_link)?$"]',
    '["waterway"~"^(canal|drain|ditch|stream|river)$"]',
    '["landuse"~"^(commercial|retail|industrial)$"]',
    '["leisure"~"^(pitch|stadium|sports_centre)$"]',
]


def overpass_near(points, radius_m=300, timeout_s=180):
    """OSM ways within radius_m of each (lat, lon) point. Needs internet."""
    parts = [f"way(around:{radius_m},{lat:.6f},{lon:.6f}){f};"
             for lat, lon in points for f in OSM_FILTERS]
    query = f"[out:json][timeout:{timeout_s}];(" + "".join(parts) + ");out geom;"
    resp = requests.post(OVERPASS_URL, data={"data": query},
                         headers={"User-Agent": "recharge-well-audit"}, timeout=timeout_s + 30)
    resp.raise_for_status()
    return resp.json()


def osm_to_layers(data):
    recs = {"roads": [], "canals": [], "drains": [], "landuse": [], "sports": []}
    for el in data.get("elements", []):
        if el.get("type") != "way" or "geometry" not in el:
            continue
        coords = [(p["lon"], p["lat"]) for p in el["geometry"]]
        if len(coords) < 2:
            continue
        tags = el.get("tags", {})
        closed = len(coords) >= 4 and coords[0] == coords[-1]
        if "highway" in tags:
            recs["roads"].append(LineString(coords))
        elif tags.get("waterway") == "canal":
            recs["canals"].append(LineString(coords))
        elif "waterway" in tags:
            recs["drains"].append(LineString(coords))
        elif "landuse" in tags and closed:
            recs["landuse"].append(Polygon(coords))
        elif "leisure" in tags and closed:
            recs["sports"].append(Polygon(coords))
    return {k: gpd.GeoDataFrame(geometry=v, crs="EPSG:4326") for k, v in recs.items()}


def level(text):
    """inside / near / unsure from a short field note."""
    t = text.lower()
    if " or " in t or "may be" in t:
        return "unsure"
    if t.startswith("in ") or t.startswith("on "):
        return "inside"
    return "near"


print("Functions loaded.")

## 2. Suitability index (the original terrain, soil and runoff layers)

Rainfall and the flood mask are not included here. The index is only used to rank cells
when a well has to move.

In [ ]:
with rasterio.open(DATA_DIR + 'Static_Predictors.tif') as src:
    stack = src.read().astype(np.float32)
    transform = src.transform

# band order of the GEE export:
# 0 slope, 1 hand, 2 twi, 3 curvature, 4 sti, 5 ndvi, 6 clay, 7 sand,
# 8 roughness, 9 curve_number, 10 infiltration_mm
slope, hand, twi = stack[0], stack[1], stack[2]
clay, sand, cn, infil = stack[6], stack[7], stack[9], stack[10]


def pct(a, q):
    return np.nanpercentile(a, q)


layers = {
    "twi": normalize(twi, pct(twi, 2), pct(twi, 98)),
    "infiltration": normalize(infil, pct(infil, 2), pct(infil, 98)),
    "slope": normalize(slope, 0, 8, invert=True),
    "hand": normalize(hand, 0, 20, invert=True),
    "soil": (normalize(sand, 0, 100) + normalize(clay, 0, 100, invert=True)) / 2,
    "curve_number": normalize(cn, 30, 100, invert=True),
}
weights = {"twi": 0.18, "infiltration": 0.18, "slope": 0.12,
           "hand": 0.12, "soil": 0.13, "curve_number": 0.12}
base = weighted_index(layers, weights)
print("Index built:", base.shape)

## 3. Original 100 wells and the first audit (200 m from mapped drains and settlements)

The drainage layer is the HydroSHEDS river network and the settlement layer is Dynamic World
built-up land. Neither contains canals or roads.

In [ ]:
wells = pd.read_csv(DATA_DIR + 'Recommended_Well_Locations_FINAL.csv')   # lon, lat
drainage = gpd.read_file(DATA_DIR + 'Faisalabad_Drainage_Network.shp')
settlements = gpd.read_file(DATA_DIR + 'Faisalabad_Settlements_Vector.shp')

audit_old = audit_sites(wells, {"drainage": drainage, "settlement": settlements},
                        buffers_m={"drainage": 200, "settlement": 200}, near_miss_m=50)
print("First audit:", summarize_audit(audit_old))
audit_old.to_csv(DATA_DIR + 'Wells_audit.csv', index=False)

## 4. Satellite check of every well (one reviewer, Google Maps satellite view)

`inside` = on a canal, drain, road or sports ground. `near` = next to one. `unsure` = the reviewer
could not tell. Everything else is `ok`. The first 30 problems were found in the first pass;
4 more (wells 5, 14, 24, 94) were found when the OpenStreetMap audit flagged them.

In [ ]:
notes_first30 = {
    31: "Near canal", 32: "In drain", 33: "Near canal", 36: "In road", 37: "In canal",
    39: "Near road", 46: "In canal", 47: "In drain", 3: "In canal", 4: "In canal",
    28: "Too close to drain and road", 54: "In drain", 57: "Near canal or in canal",
    59: "Near drain", 61: "In cricket ground", 62: "Near road", 67: "Close to road",
    68: "In canal", 72: "May be in canal or drain", 73: "On the road", 74: "In drain",
    77: "In drain", 79: "On the road", 80: "In drain or on road",
    87: "Near motorway and road", 90: "In canal", 92: "In drain",
    97: "Near Gogera Branch Canal", 98: "Near drain", 99: "In canal",
}
notes_after_osm = {
    5: "Near canal or may be drain, also near paper and board industry",
    14: "Near Shahkot minor", 24: "May be canal or drain",
    94: "May be near canal or drain",
}
notes = {**notes_first30, **notes_after_osm}

chk = audit_old.copy()
chk['well_id'] = chk.index
chk['observed'] = 'ok'
chk['notes'] = ''
for k, v in notes.items():
    chk.loc[chk['well_id'] == k, 'observed'] = level(v)
    chk.loc[chk['well_id'] == k, 'notes'] = v
chk['map_link'] = ("https://www.google.com/maps/search/?api=1&query="
                   + chk['lat'].astype(str) + "," + chk['lon'].astype(str))
chk.to_csv(DATA_DIR + 'Wells_field_check.csv', index=False)

print(chk['observed'].value_counts().to_string())
print("\nFirst audit label (rows) against a problem seen by eye (columns):")
print(pd.crosstab(chk['label'], chk['observed'] != 'ok', margins=True))

### Does the index tell problem wells from fine wells?

If it did, problem wells would score differently. They do not.

In [ ]:
wells['col'] = ((wells['lon'] - transform.c) / transform.a).astype(int)
wells['row'] = ((wells['lat'] - transform.f) / transform.e).astype(int)
vals = pd.DataFrame({n: np.asarray(layers[n])[wells['row'], wells['col']] for n in layers})
vals['observed'] = chk['observed'].to_numpy()
print("Mean layer value at the wells (1.0 = best), by what was seen by eye:")
print(vals.groupby('observed').mean().round(3).T.to_string())

score = base[wells['row'], wells['col']]
valid = np.isfinite(base)
print("\nIndex score at the wells: min", round(float(score.min()), 3),
      "max", round(float(score.max()), 3))
print("Share of the district scoring at least as high as the lowest well:",
      round(100 * float((base[valid] >= score.min()).mean()), 3), "%")

## 5. OpenStreetMap audit (needs internet)

Roads 30 m, canals 50 m, drains 50 m. `landuse` and `sports` returned no features here, so
commercial land and sports grounds cannot be caught from this data.

In [ ]:
data = overpass_near(list(zip(wells['lat'], wells['lon'])))
osm = osm_to_layers(data)
print("OSM features found:", {k: len(v) for k, v in osm.items()})
osm = {k: v for k, v in osm.items() if len(v) > 0}   # an empty layer means 'no data'

buffers = {"roads": 30, "canals": 50, "drains": 50, "landuse": 1, "sports": 1}
buffers = {k: v for k, v in buffers.items() if k in osm}

a2 = audit_sites(wells, osm, buffers_m=buffers, near_miss_m=0)
a2['field'] = chk['observed'].to_numpy()
print("\nOSM audit (rows) against what was seen by eye (columns):")
print(pd.crosstab(a2['field'], a2['label'], margins=True))
a2.to_csv(DATA_DIR + 'Wells_audit_OSM.csv', index=False)

problem = (chk['observed'].to_numpy() != 'ok')
rows = []
for scale in [0.5, 1, 1.5, 2, 3]:
    b = {k: (v if v <= 1 else v * scale) for k, v in buffers.items()}
    a = audit_sites(wells, osm, buffers_m=b, near_miss_m=0)
    flagged = (a['label'] == 'conflict').to_numpy()
    rows.append({"buffers_m": b,
                 f"caught_of_{int(problem.sum())}": int((flagged & problem).sum()),
                 f"false_alarms_of_{int((~problem).sum())}": int((flagged & ~problem).sum())})
print("\nBuffer trade-off:")
print(pd.DataFrame(rows).to_string(index=False))

## 6. Round 1: move the wells found to be a problem in the first check

Each problem spot is also blocked as a 300 m circle (plus a 50 m margin), because a drain or
canal that is missing from OpenStreetMap continues beyond the point where it was seen.

In [ ]:
first_ids = sorted(notes_first30)
wells_all = pd.read_csv(DATA_DIR + 'Wells_audit.csv')
wells_all['well_id'] = wells_all.index
to_move = wells_all[wells_all['well_id'].isin(first_ids)]
staying = wells_all[~wells_all['well_id'].isin(first_ids)]

confirmed = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(to_move['lon'], to_move['lat']), crs="EPSG:4326")
constraints = {**osm, "drainage": drainage, "settlement": settlements,
               "seen_problem": confirmed}
buffers_all = {**buffers, "drainage": 200, "settlement": 200, "seen_problem": 300}

moves = relocate_sites(to_move, base, transform, constraints=constraints,
                       buffers_m=buffers_all, other_sites=staying)
moves['well_id'] = to_move['well_id'].to_numpy()
print(moves['status'].value_counts().to_string())
print("Distance moved (m): min", round(float(moves['moved_m'].min())),
      "median", round(float(moves['moved_m'].median())),
      "max", round(float(moves['moved_m'].max())))
moves.to_csv(DATA_DIR + 'Wells_relocation.csv', index=False)

final = wells_all[['well_id', 'lon', 'lat']].copy()
final['history'] = 'original'
for _, r in moves[moves['status'] != 'no_alternative'].iterrows():
    m = final['well_id'] == r['well_id']
    final.loc[m, ['lon', 'lat']] = [r['new_lon'], r['new_lat']]
    final.loc[m, 'history'] = 'moved_round1'
final.to_csv(DATA_DIR + 'Wells_final_v2.csv', index=False)
print("Moved in round 1:", int((final['history'] == 'moved_round1').sum()))

## 7. Round 2: the new locations were checked on satellite again

Wells that still had a problem after round 1 (including 5, 14, 24 and 94, which were not moved
in round 1). `manual_lines` lets you add drains, canals or roads you can see but that are
missing from OpenStreetMap, as lists of (lat, lon) points; a single point is blocked as a circle.

In [ ]:
problems_after_round1 = {
    47: "Near drain", 59: "Uncertain, may be near drain", 61: "Near road",
    67: "In empty plot near houses and fields", 92: "Near drain",
    97: "Near Gogera Branch Canal", 54: "In the drain", 28: "Near drain",
    32: "Near the drain", 74: "In the drain", 87: "Near motorway",
    90: "May be in canal or drain",
    5: "Near canal or may be drain, also near paper and board industry",
    14: "Near Shahkot minor", 24: "May be canal or drain",
    94: "May be near canal or drain",
}
manual_lines = {
    # "drain_near_47": [(31.4524, 73.3912), (31.4531, 73.3950), (31.4540, 73.3990)],
}
line_buffer_m = 150

ids_again = sorted(problems_after_round1)
final = pd.read_csv(DATA_DIR + 'Wells_final_v2.csv')
to_move2 = final[final['well_id'].isin(ids_again)]
staying2 = final[~final['well_id'].isin(ids_again)]

geoms = [Point(pts[0][1], pts[0][0]) if len(pts) == 1
         else LineString([(lon, lat) for lat, lon in pts]) for pts in manual_lines.values()]
manual = gpd.GeoDataFrame(geometry=geoms, crs="EPSG:4326")
still = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(to_move2['lon'], to_move2['lat']), crs="EPSG:4326")

constraints2 = {**osm, "drainage": drainage, "settlement": settlements,
                "your_features": manual, "seen_problem": still}
buffers2 = {**buffers, "drainage": 200, "settlement": 200,
            "your_features": line_buffer_m, "seen_problem": 300}

moves2 = relocate_sites(to_move2, base, transform, constraints=constraints2,
                        buffers_m=buffers2, other_sites=staying2,
                        search_radius_m=3000, keep_apart_m=1500)
moves2['well_id'] = to_move2['well_id'].to_numpy()
moves2['check_first'] = moves2['moved_m'] > 1000      # far moves were unreliable in round 1
moves2['map_link'] = ("https://www.google.com/maps/search/?api=1&query="
                      + moves2['new_lat'].astype(str) + "," + moves2['new_lon'].astype(str))
moves2.to_csv(DATA_DIR + 'Wells_relocation_round2.csv', index=False)
print(moves2['status'].value_counts().to_string())

## 8. Round 2 result and the final list

Wells still near a drain, canal or houses after round 2 are marked `check_in_field`.

In [ ]:
problems_after_round2 = {
    28: "Near drain", 47: "Near drain", 59: "Near drain",
    61: "In the drain or may be in the canal, near a house",
    87: "Near houses and shops",
}

for _, r in moves2[moves2['status'] != 'no_alternative'].iterrows():
    m = final['well_id'] == r['well_id']
    final.loc[m, ['lon', 'lat']] = [r['new_lon'], r['new_lat']]
    final.loc[m, 'history'] = 'moved_round2'
final['status'] = np.where(final['well_id'].isin(list(problems_after_round2)),
                           'check_in_field', 'no_visible_conflict')
final['map_link'] = ("https://www.google.com/maps/search/?api=1&query="
                     + final['lat'].astype(str) + "," + final['lon'].astype(str))
final.to_csv(DATA_DIR + 'Wells_final.csv', index=False)

n = len(final)
summary = pd.DataFrame({
    "step": ["Index selection", "After round 1", "After round 2"],
    "wells_without_visible_problem": [n - len(notes), n - len(problems_after_round1),
                                      n - len(problems_after_round2)],
})
print(summary.to_string(index=False))
print()
print(final['status'].value_counts().to_string())
print(final['history'].value_counts().to_string())

## 9. Figures for the README and LinkedIn

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4))
bars = ax.bar(summary['step'], summary['wells_without_visible_problem'],
              color=['#b2182b', '#ef8a62', '#2166ac'])
ax.bar_label(bars, fmt='%d', fontsize=12, fontweight='bold')
ax.set_ylim(0, n)
ax.set_ylabel(f'Wells with no visible problem (of {n})')
ax.set_title('Candidate recharge wells: practical audit, Faisalabad')
plt.tight_layout()
plt.savefig(DATA_DIR + 'fine_wells_before_after.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.patches as mpatches

boundary_raw = gpd.read_file(DATA_DIR + 'Faisalabad_Boundary.shp')
outline = boundary_raw.dissolve()
tehsils = boundary_raw.dissolve(by='adm3_pcode') if 'adm3_pcode' in boundary_raw.columns else None
drain_plot = gpd.clip(drainage.to_crs(outline.crs), outline)

fin = pd.read_csv(DATA_DIR + 'Wells_final.csv')
unchanged = fin[(fin['history'] == 'original') & (fin['status'] == 'no_visible_conflict')]
moved_ok = fin[(fin['history'] != 'original') & (fin['status'] == 'no_visible_conflict')]
needs_check = fin[fin['status'] == 'check_in_field']

fig, ax = plt.subplots(figsize=(11, 11))
outline.boundary.plot(ax=ax, color='black', linewidth=1.4, zorder=3)
if tehsils is not None:
    tehsils.boundary.plot(ax=ax, color='dimgray', linewidth=0.6, linestyle='--', zorder=2)
drain_plot.plot(ax=ax, color='#6baed6', linewidth=0.5, alpha=0.7, zorder=1)

# original problem sites and the way they moved (round 1 old -> new)
try:
    r1 = pd.read_csv(DATA_DIR + 'Wells_relocation.csv')
    r1 = r1[r1['status'] != 'no_alternative']
    for _, r in r1.iterrows():
        ax.plot([r['old_lon'], r['new_lon']], [r['old_lat'], r['new_lat']],
                color='gray', linewidth=0.6, zorder=4)
    ax.scatter(r1['old_lon'], r1['old_lat'], marker='x', s=22, c='#b2182b',
               linewidths=0.9, zorder=5)
except FileNotFoundError:
    pass

ax.scatter(unchanged['lon'], unchanged['lat'], s=26, c='#2166ac', edgecolors='white',
           linewidths=0.4, zorder=6)
ax.scatter(moved_ok['lon'], moved_ok['lat'], s=70, marker='*', c='#1a9850',
           edgecolors='black', linewidths=0.4, zorder=7)
ax.scatter(needs_check['lon'], needs_check['lat'], s=60, marker='D', c='#ff7f00',
           edgecolors='black', linewidths=0.6, zorder=8)

minx, miny, maxx, maxy = outline.total_bounds
ax.set_xlim(minx, maxx)
ax.set_ylim(miny, maxy)
ax.set_aspect(1 / np.cos(np.deg2rad((miny + maxy) / 2)))

# north arrow and scale bar
ax.annotate('N', xy=(0.94, 0.86), xytext=(0.94, 0.93), xycoords='axes fraction',
            ha='center', va='center', fontsize=15, fontweight='bold',
            arrowprops=dict(facecolor='black', width=3, headwidth=11, headlength=11))
km_per_deg = 111.32 * np.cos(np.deg2rad((miny + maxy) / 2))
bar = 15 / km_per_deg
x0, y0 = minx + 0.05 * (maxx - minx), miny + 0.05 * (maxy - miny)
ax.plot([x0, x0 + bar], [y0, y0], color='black', linewidth=3, zorder=9)
ax.text(x0 + bar / 2, y0 + 0.008 * (maxy - miny), '15 km', ha='center', fontsize=10,
        fontweight='bold')

handles = [
    mpatches.Patch(color='#2166ac', label=f'Unchanged, no visible conflict ({len(unchanged)})'),
    plt.Line2D([], [], marker='*', color='w', markerfacecolor='#1a9850',
               markeredgecolor='black', markersize=13, label=f'Moved, no visible conflict ({len(moved_ok)})'),
    plt.Line2D([], [], marker='D', color='w', markerfacecolor='#ff7f00',
               markeredgecolor='black', markersize=8, label=f'Needs a field check ({len(needs_check)})'),
    plt.Line2D([], [], marker='x', color='#b2182b', linestyle='None',
               label='Original problem site (round 1)'),
]
ax.legend(handles=handles, loc='lower right', fontsize=9, framealpha=0.95)
ax.set_title('Candidate recharge wells after a practical audit, Faisalabad District',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.text(0.01, 0.01, 'Roads, canals, drains: OpenStreetMap contributors (ODbL). '
        'One reviewer, satellite imagery only.', transform=ax.transAxes, fontsize=7.5,
        va='bottom')
plt.tight_layout()
plt.savefig(DATA_DIR + 'Faisalabad_final_wells_map.png', dpi=300, bbox_inches='tight',
            facecolor='white')
plt.show()
try:
    from google.colab import files
    files.download(DATA_DIR + 'Faisalabad_final_wells_map.png')
except Exception:
    pass
print("Saved Faisalabad_final_wells_map.png")

## Limitations

- One reviewer, satellite imagery only. "Near" is a visual judgement.
- Round 1 used the reviewer's list of problem wells to decide which wells to move. The OpenStreetMap
  audit alone would have flagged 30 wells, 29 of them real problems.
- OpenStreetMap does not contain every drain or sports ground, so "clear" only means no conflict
  was found in the layers used. Moves of more than 1 km were unreliable in round 1.
- The index is saturated in this flat plain: at the selected wells the terrain layers are at their
  maximum and scores differ by about 0.004, so the exact spot inside the best area is largely arbitrary.
- No groundwater quality, water source, land ownership or infiltration test is used. Not validated in
  the field. Not a groundwater model.